# Aula 1 — A bancada e o modelo de velocidade

**Projeto final · Laboratório de Controle Automático · Ifes Guarapari · 11/11 · 1h40**

**Objetivo da aula:** deixar a bancada funcionando e obter o primeiro modelo matemático do motor
— como a **velocidade** responde ao PWM. Esse modelo é a primeira metade do **Trabalho 1**.

**Ao final desta aula a bancada terá:** o firmware gravado e testado · a calibração do encoder
(contas por grau) · o modelo de velocidade FOPDT ($K$, $\tau$, $L$, $u_0$) salvo.

**Papéis na bancada (troquem a cada aula):** 🔧 **operador(a)** — mexe no hardware e grava o
firmware · 💻 **programador(a)** — roda e edita o notebook · ✍️ **relator(a)** — escreve as
respostas ❓ e confere os números. Com 2 pessoas, o relator acumula com o programador.

**Legenda:** ⏱️ tempo sugerido · 📘 teoria (leia antes de fazer) · 🔧 bancada · 💻 notebook ·
✏️ complete o código · ❓ responda · ✅ checkpoint — **chame o professor**.

| ⏱️ | Parte | O que se faz |
|---|---|---|
| 0–10 min | 0 | O projeto em uma página |
| 10–30 min | 1 | Gravar o firmware e testar o sentido de giro |
| 30–40 min | 2 | Calibrar o encoder |
| 40–55 min | 3 | Ensaio PRBS |
| 55–90 min | 4 | Ajustar o modelo FOPDT |
| 90–100 min | ✅ | Checkpoint 1 |

## Parte 0 — O projeto em uma página · ⏱️ 10 min

Vamos construir um **servo de posição**: o Arduino mede o ângulo da roda e comanda o motor para
que ela siga uma referência — 0° → 90° → −90° → 0°, mudando a cada 5 s. Para o degrau de 90°:

| # | Especificação | Valor |
|---|---|---|
| 1 | erro de regime | $|e_{ss}| < 2°$ |
| 2 | sobressinal | $M_p \le 15$ % |
| 3 | tempo de acomodação (±5 %) | $t_s \le 2{,}5$ s |
| 4 | sem *windup* | com o PWM saturado, o sobressinal cresce no máximo 5 pontos percentuais |

| Aula | Data | O que a bancada faz | Vale para |
|---|---|---|---|
| 1 | 11/11 | modelo de velocidade (hoje) | Trabalho 1 |
| 2 | 18/11 | modelo de posição, atrito, 1º projeto pelo lugar das raízes | Trabalho 1 |
| 3 | 25/11 | PID pelo lugar das raízes, relé, compensação do atrito | Trabalho 2 |
| 4 | 03/12 | ensaios finais e anti-windup | Trabalho 2 |
| 5 | 10/12 | fechamento e apresentação | Projeto final |

Cada aula é um notebook como este. Tudo o que vocês precisam de teoria está nas caixas 📘, no
momento em que é usado. **Salvem o notebook (Ctrl+S) ao final de cada parte.**

## Parte 1 — Gravar o firmware e testar o sentido de giro · ⏱️ 20 min

O firmware `pid_motor.ino` é gravado **uma vez só**. Daqui em diante, todo ensaio é configurado
pelo notebook: a função `coletar()` abre a porta serial (isso reinicia o Arduino), envia a
configuração do ensaio, recebe as medições a cada 20 ms e salva um arquivo CSV na pasta `dados/`.

> 🖥️ **Onde rodar.** Este notebook roda **no desktop da bancada** (Ubuntu), não no Google Colab:
> o Colab roda num servidor do Google, que não enxerga a porta USB do Arduino. Se você está lendo
> isto no Jupyter do desktop, o ambiente já está pronto. Se não estiver, siga
> `projeto_final/INSTALACAO_UBUNTU.md` (≈ 10 min, com o script `instalar_ubuntu.sh`).
> Para abrir o Jupyter: Terminal → `cd ~/lab-cont-2026-2/projeto_final` → `controle` →
> `jupyter notebook`.

**🔧 Passo 1.1 — gravar o firmware**
1. Ligue o cabo USB e a fonte do motor.
2. Abra **outra aba** do Terminal (Ctrl+Shift+T) — a do Jupyter tem que continuar aberta — e
   digite:
   ```bash
   cd ~/lab-cont-2026-2/projeto_final
   bash gravar_firmware.sh
   ```
   O script acha a porta, compila e grava (≈ 1 min). No fim ele imprime a porta usada
   (ex.: `PORTA = "/dev/ttyUSB0"`) e o display mostra **PRONTO**.
   - `Sem permissão para usar /dev/ttyUSB0` ou `Nenhum Arduino encontrado`: veja a tabela da
     Seção 4 do `INSTALACAO_UBUNTU.md` (grupo `dialout`, `brltty`).
   - Prefere a Arduino IDE? **Arquivo › Abrir** → `firmware/pid_motor/pid_motor.ino`, placa
     **Arduino Uno**, porta `/dev/ttyUSB0`, **Carregar**.
3. A porta só aceita um programa de cada vez: **feche a Arduino IDE / Monitor Serial** antes de
   usar o notebook.

**💻 Passo 1.2 —** confira a porta e escreva o número da bancada na célula abaixo; rode-a. Ela
lista os Arduinos conectados — use o nome que aparecer.

In [ ]:
# ⚙️ CONFIGURAÇÃO — rode esta célula sempre que abrir o notebook
import numpy as np
import matplotlib.pyplot as plt
import control as ct
from scipy.optimize import minimize
from ferramentas_projeto import *

PORTA = "/dev/ttyUSB0"   # ✏️ porta do Arduino (veja a lista impressa abaixo; Uno original: "/dev/ttyACM0")
BANCADA = "1"            # ✏️ número da bancada

s = ct.tf("s")
Ts = 0.02          # período de amostragem do firmware [s]
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})
P = carregar_parametros()          # resultados das aulas anteriores (dados/parametros_bancada.json)
print("bancada", BANCADA, "| parâmetros já salvos:", list(P))
print("Arduinos conectados:"); portas()

**🔧💻 Passo 1.3 — primeiro ensaio: degrau de PWM** (MODO 1). A roda vai girar por 2 s com
PWM = 100 (de 255). **Deixe a roda livre — nada pode encostar nela.**

In [ ]:
d = coletar("teste_sentido", PORTA, MODO=1, PWM_MA=100)
plt.figure(figsize=(8, 3))
plt.plot(d["t"], d["angle"])
plt.xlabel("tempo (s)"); plt.ylabel("posição (contas do encoder)")
plt.title("Teste de sentido: com PWM positivo, a posição tem que CRESCER"); plt.show()
print("posição final:", d["angle"][-1], "contas")

❓ **1.1** A posição cresceu? (Se **diminuiu**: desligue a fonte, inverta os dois fios do
motor nos bornes OUT1/OUT2 da ponte H e repita o Passo 1.3. **Não** troque os fios do encoder.)

✍️ **Resposta da bancada:** *(escreva aqui)*

## Parte 2 — Calibrar o encoder · ⏱️ 10 min

📘 **Encoder.** Atrás do motor há um disco magnético e dois sensores Hall (canais C1 e C2). Cada
pulso de C1 é uma fração de volta do eixo do motor; como há um **redutor** entre o motor e a roda,
uma volta da roda gera centenas de pulsos. O firmware conta as **duas bordas** (subida e descida)
de C1 e usa C2 para saber o sentido. Precisamos saber quantas **contas** correspondem a 1 grau
da roda — o número `CPG` (contas por grau). A **resolução** é $1/\text{CPG}$ graus: nenhuma
medida de ângulo será mais fina que isso.

**🔧 Passo 2.1** — faça uma marca na roda (fita ou caneta) e outra na base, alinhadas.

**🔧💻 Passo 2.2** — rode a célula abaixo. Quando aparecer a linha `# MODO=0 ...`, **gire a roda
com a mão, devagar, exatamente 5 voltas** no sentido em que ela girou no teste (vocês têm 20 s).

In [ ]:
cal = coletar("calibracao", PORTA, MODO=0, T=20000)
N_VOLTAS = 5
contas = cal["u_pid"][-1]          # no MODO 0, a 5ª coluna traz as contas brutas
CPG = ...                          # ✏️ contas por grau = contas / (360 × número de voltas)
print(f"contas em {N_VOLTAS} voltas = {contas:.0f}  ->  CPG = {CPG:.4f} contas/grau")
print(f"resolução = {1/CPG:.3f} grau por conta")
P = salvar_parametros(CPG=CPG)

❓ **2.1** Quantas contas o encoder dá por volta da roda? Qual a resolução em graus? Quantas
contas "cabem" na tolerância de 2° da especificação 1?

✍️ **Resposta da bancada:** *(escreva aqui)*

## Parte 3 — Ensaio PRBS · ⏱️ 15 min

📘 **Identificação de sistemas.** Em vez de deduzir o modelo das leis físicas (resistência,
indutância, inércia, atrito — que não conhecemos), aplicamos uma entrada conhecida, medimos a
saída e **ajustamos** um modelo que reproduza a medição.

📘 **PRBS** (*pseudo-random binary sequence*): o PWM pula entre dois níveis em instantes
"aleatórios" (mas reproduzíveis), múltiplos de um período de bit $T_b$. Isso excita o motor em
muitas frequências ao mesmo tempo — com um degrau só, veríamos pouco. Três escolhas do ensaio:
- **níveis 60 ↔ 220** (= 140 ± 80): o motor **nunca para**. Perto de PWM 0 o atrito "come" o
  comando e o motor deixa de se comportar como um sistema linear (veremos isso na Parte 4);
- **$T_b$ = 200 ms**: ≈ 2 vezes a constante de tempo esperada (≈ 0,1 s), para a velocidade ter
  tempo de quase se acomodar em cada nível;
- **saída = contas por janela de 20 ms**: é uma medida de **velocidade** (5ª coluna no MODO 5).

**🔧💻 Passo 3.1** — rode o ensaio (≈ 35 s; a roda vai acelerar e desacelerar).

In [ ]:
prbs = coletar("prbs", PORTA, MODO=5, CPG=P["CPG"], TB=200, T=30000)
t1, u1, v1 = prbs["t"], prbs["pwm"], prbs["u_pid"]     # v1 = contas por janela de 20 ms
fig, ax = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
ax[0].step(t1, u1, "k", where="post"); ax[0].set_ylabel("PWM")
ax[1].plot(t1, v1, ".-", ms=3); ax[1].set_ylabel("velocidade\n(contas/janela)")
ax[1].set_xlabel("tempo (s)"); ax[1].set_xlim(0, 6)
ax[0].set_title("PRBS — primeiros 6 s"); plt.show()

❓ **3.1** Olhe uma subida da velocidade logo após o PWM passar de 60 para 220. Em quanto tempo
ela chega a ≈ 63 % da variação total? (Esse tempo é a **constante de tempo** $\tau$.) Escreva sua
estimativa — ela será o "chute inicial" do ajuste na Parte 4.

✍️ **Resposta da bancada:** *(escreva aqui)*

## Parte 4 — Ajustar o modelo FOPDT · ⏱️ 35 min

📘 **Modelo de 1ª ordem com atraso (FOPDT):**
$$\frac{V(s)}{U(s)} = \frac{K\,e^{-Ls}}{\tau s + 1}$$
- $K$ — **ganho**: quanto a velocidade varia, em regime, por unidade de PWM [(contas/janela)/PWM];
- $\tau$ — **constante de tempo**: tempo para chegar a 63 % da variação [s];
- $L$ — **atraso puro**: tempo até a saída começar a reagir [s]; $e^{-Ls}$ é o atraso no domínio
  de Laplace.

📘 **Ajuste por mínimos quadrados.** Para um palpite $(K, \tau, L)$, simulamos o modelo com o
**mesmo** PWM do ensaio e calculamos o erro quadrático médio (EQM) entre a velocidade simulada e
a medida. Um otimizador (Nelder-Mead) muda os parâmetros até o EQM ser mínimo. A função
`ajusta_fopdt()` faz isso.

📘 **Desvio de entrada $u_0$ (zona morta).** O motor só começa a girar acima de algumas dezenas de
PWM: parte do comando é "gasta" vencendo o atrito. Um modelo mais honesto é
$$V(s) = \frac{K\,e^{-Ls}}{\tau s + 1}\,\big(U(s) - u_0\big),$$
em que $u_0$ é, aproximadamente, a **zona morta**. Vamos ajustar as **duas** versões e comparar.

**💻 Passo 4.1** — escreva o seu chute para $\tau$ (Parte 3) e rode.

In [ ]:
chute_tau = ...                     # ✏️ sua estimativa de τ em segundos (pergunta 3.1)
chute_K = (v1.max() - v1.min()) / (u1.max() - u1.min())     # variação da saída / variação da entrada
p_sem, eqm_sem = ajusta_fopdt(t1, u1, v1, [chute_K, chute_tau, 0.02])
p_com, eqm_com = ajusta_fopdt(t1, u1, v1, [chute_K, chute_tau, 0.02, 30])
print("                     K          tau [s]   L [ms]   u0 [PWM]   EQM")
print(f"sem desvio   {p_sem[0]:10.4f} {p_sem[1]:10.3f} {1e3*p_sem[2]:8.1f}        —   {eqm_sem:8.3f}")
print(f"com desvio   {p_com[0]:10.4f} {p_com[1]:10.3f} {1e3*p_com[2]:8.1f} {p_com[3]:8.1f}   {eqm_com:8.3f}")

In [ ]:
plt.figure(figsize=(10, 3.8))
plt.plot(t1, v1, color="0.6", lw=3, label="medido")
plt.plot(t1, simula_fopdt(p_sem, t1, u1), "r--", label="FOPDT sem desvio")
plt.plot(t1, simula_fopdt(p_com, t1, u1), "b", label="FOPDT com desvio $u_0$")
plt.xlim(4, 10); plt.xlabel("tempo (s)"); plt.ylabel("velocidade (contas/janela)")
plt.legend(); plt.title("Medido × modelos"); plt.show()

❓ **4.1** Compare os dois ajustes: qual tem menor EQM? Qual $K$ é maior? Por que o modelo
**sem** desvio erra o ganho, sabendo que o PRBS nunca leva o PWM abaixo de 60?

✍️ **Resposta da bancada:** *(escreva aqui)*

❓ **4.2** O que o valor de $u_0$ significa fisicamente? Confira: com PWM igual a $u_0$, a que
velocidade o modelo leva o motor?

✍️ **Resposta da bancada:** *(escreva aqui)*

❓ **4.3** O atraso $L$ saiu da ordem de 1 período de amostragem (20 ms). De onde ele vem?
Dica: a velocidade é medida contando os pulsos **durante** uma janela de 20 ms.

✍️ **Resposta da bancada:** *(escreva aqui)*

📘 **Padé (para saber o que é — vamos usar na Aula 2).** O atraso $e^{-Ls}$ não é uma razão de
polinômios, e as ferramentas de lugar das raízes e de margens precisam de polinômios. A
aproximação de **Padé** de 1ª ordem troca o atraso por um polo e um zero:
$$e^{-Ls} \approx \frac{1 - Ls/2}{1 + Ls/2}$$
— repare no **zero no semiplano direito** ($s = +2/L$): é ele que imita o "começo atrasado" da
resposta. Para $L$ pequeno, como o nosso, ela é uma boa aproximação. A célula abaixo compara.

In [ ]:
K, tau, L, u0 = p_com
G1 = ct.tf([K], [tau, 1]) * ct.tf(*ct.pade(L, 1))
_, v_pade = ct.forced_response(G1, t1 - t1[0], u1 - u0)
print(f"EQM com atraso exato: {eqm_com:.3f}   |   com Padé de 1ª ordem: {np.mean((v1 - v_pade)**2):.3f}")
P = salvar_parametros(K=K, tau=tau, L=L, u0=u0)

## ✅ Checkpoint 1 · ⏱️ 10 min

Rode a célula abaixo, **salve o notebook** e chame o professor. Ele vai conferir o teste de
sentido, o `CPG` e o ajuste do FOPDT.

In [ ]:
P = carregar_parametros()
print(f"Bancada {BANCADA} — resultados da Aula 1")
print(f"  CPG = {P['CPG']:.4f} contas/grau  (resolução {1/P['CPG']:.3f}°)")
print(f"  FOPDT da velocidade: K = {P['K']:.4f} (contas/janela)/PWM, tau = {P['tau']:.3f} s, "
      f"L = {1e3*P['L']:.1f} ms, u0 = {P['u0']:.1f} PWM")